# Imports

In [2]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, load_metric_from_log

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying loss

## load data

In [178]:
root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/results_ML3/diff_loss'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle', 'auxi_mode', 'auxi_type', 'rank_ratio']
metric_names = ['mse', 'mae', 'cov']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        # shutil.rmtree(exp_dir)
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    if len(metric) == 6:
        log_metrics = load_metric_from_log(os.path.join(exp_dir, 'result_long_term_forecast.txt'))
        cov_loss = log_metrics['cov'] if log_metrics and 'cov' in log_metrics else np.inf
        result.loc[:, metric_names] = metric[1], metric[0], cov_loss
    else:
        result.loc[:, metric_names] = metric[1], metric[0], metric[2]
    result.loc[:, ['meta_type']] = config[['meta_type']] if 'meta_type' in config.columns else 'all'
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/varying_loss_trans.csv", index=False)

df.head(4)

,model,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,auxi_mode,auxi_type,rank_ratio,mse,mae,cov,meta_type,exp_dir
71,PDF,96,ECL,0.0002,0.0005,0.0005,0.001,0.999,0.0001,type1,100,10,16,1024,20,1,0.15,5,1.0,MAE,1,0.1,24,basis,pca,1.0,0.172785,0.252711,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
72,PDF,96,ECL,0.0005,0.0005,0.0005,0.001,0.999,0.0001,type1,100,10,16,1024,20,1,0.15,5,1.0,MAE,1,0.1,24,basis,pca,1.0,0.164910,0.246471,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
73,PDF,96,ECL,0.0002,0.0005,0.0005,0.020,0.980,0.0001,type1,100,10,16,1024,20,1,0.15,5,1.0,MAE,1,0.1,24,basis,pca,1.0,0.171917,0.252241,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
74,PDF,96,ECL,0.0002,0.0005,0.0005,0.020,0.980,0.0001,type1,100,10,16,1024,20,1,0.15,5,1.0,MAE,1,0.1,24,rfft,complex,1.0,0.171367,0.252001,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


In [179]:
params = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle', 'auxi_mode', 'auxi_type', 'rank_ratio']


df[(df.model == 'TQNet') & (df.data_id == 'ETTm1') & (df.auxi_mode == 'basis')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'TQNet') & (df.data_id == 'ETTm1') & (df.auxi_mode == 'rfft')].sort_values(by=['pred_len', 'mse'])[params]


# df[(df.model == 'TQNet') & (df.data_id == 'ETTh1') & (df.auxi_mode == 'basis')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'TQNet') & (df.data_id == 'ETTh1') & (df.auxi_mode == 'rfft')].sort_values(by=['pred_len', 'mse'])[params]


# df[(df.model == 'PDF') & (df.data_id == 'ETTh1') & (df.auxi_mode == 'basis')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'PDF') & (df.data_id == 'ETTh1') & (df.auxi_mode == 'rfft')].sort_values(by=['pred_len', 'mse'])[params]


# df[(df.model == 'PDF') & (df.data_id == 'ECL') & (df.auxi_mode == 'basis')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'PDF') & (df.data_id == 'ECL') & (df.auxi_mode == 'rfft')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'PDF') & (df.data_id == 'ECL') & (df.auxi_mode == 'fourier_koopman')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'PDF') & (df.data_id == 'ECL') & (df.auxi_mode == 'dilate_cuda')].sort_values(by=['pred_len', 'mse'])[params]



# df[(df.model == 'PDF') & (df.data_id == 'Weather') & (df.auxi_mode == 'basis')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'PDF') & (df.data_id == 'Weather') & (df.auxi_mode == 'rfft')].sort_values(by=['pred_len', 'mse'])[params]
# df[(df.model == 'PDF') & (df.data_id == 'Weather') & (df.auxi_mode == 'dilate_cuda')].sort_values(by=['pred_len', 'mse'])[params]


,model,pred_len,data_id,mse,mae,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,auxi_mode,auxi_type,rank_ratio
559,TQNet,96,ETTm1,0.305506,0.343767,0.0005,0.0005,0.0005,0.0,1.0,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
563,TQNet,96,ETTm1,0.306845,0.347260,0.0010,0.0005,0.0005,0.1,0.9,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
560,TQNet,96,ETTm1,0.306930,0.345565,0.0010,0.0005,0.0005,0.0,1.0,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
565,TQNet,96,ETTm1,0.307039,0.348723,0.0005,0.0005,0.0005,0.2,0.8,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
561,TQNet,96,ETTm1,0.307491,0.348137,0.0002,0.0005,0.0005,0.1,0.9,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
566,TQNet,96,ETTm1,0.307946,0.348437,0.0010,0.0005,0.0005,0.2,0.8,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
562,TQNet,96,ETTm1,0.308942,0.348337,0.0005,0.0005,0.0005,0.1,0.9,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
564,TQNet,96,ETTm1,0.309419,0.351388,0.0002,0.0005,0.0005,0.2,0.8,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
558,TQNet,96,ETTm1,0.309608,0.345973,0.0002,0.0005,0.0005,0.0,1.0,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0
503,TQNet,192,ETTm1,0.352860,0.374950,0.0010,0.0005,0.0005,0.2,0.8,0.0001,type1,30,5,32,1024,20,1,0.15,5,1.0,MAE,1,0.5,96,basis,pca,1.0


## preprocess

In [186]:
stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
log_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'

baselines = pd.read_csv(f'{log_root}/baselines_chosen.csv')
finetunes = pd.read_csv(f'{stats_root}/finetune_best.csv')

df = pd.read_csv(f'{stats_root}/varying_loss_trans.csv')

best = finetunes.copy()
best = best[best.data_id.isin(['ETTm1', 'ETTh1', 'ECL', 'Weather'])]
best = best[best.model.isin(['TQNet', 'PDF'])]
best['auxi_mode'] = 'QDF'

base = baselines.copy()
base = base[base.data_id.isin(['ETTm1', 'ETTh1', 'ECL', 'Weather'])]
base = base[base.model.isin(['TQNet', 'PDF'])]
base['auxi_mode'] = 'DF'


df2 = df.copy()
df2 = df2[df2.model.isin(['TQNet', 'PDF'])]
# min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len', 'auxi_mode'])['mse'].idxmin()
# df2 = df2.loc[min_mse_idx]

df2_m1_tqnet_basis = df2[
    (df2.data_id.isin(['ETTm1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'basis') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.8) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ETTm1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'basis') & df2.pred_len.isin([192, 336, 720]))
]

df2_m1_tqnet_rfft = df2[
    (df2.data_id.isin(['ETTm1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.1) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ETTm1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.2) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ETTm1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'rfft') & df2.pred_len.isin([336, 720]))
]

df2_h1_tqnet_basis = df2[
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'basis') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.8) & (df2.learning_rate == 0.0001)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'basis') & df2.pred_len.isin([192, 336, 720]))
]
df2_h1_tqnet_rfft = df2[
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.2) & (df2.learning_rate == 0.001)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'TQNet') & (df2.auxi_mode == 'rfft') & df2.pred_len.isin([192, 336, 720]))
]

df2_tqnet_others = df2[
    ((df2.data_id.isin(['ECL', 'Weather'])) & (df2.model == 'TQNet')) |
    (df2.data_id.isin(['ETTm1']) & (df2.model == 'TQNet') & (df2.auxi_mode != 'basis') & (df2.auxi_mode != 'rfft')) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'TQNet') & (df2.auxi_mode != 'basis') & (df2.auxi_mode != 'rfft'))
]
df2_tqnet = pd.concat([
    df2_tqnet_others,
    df2_m1_tqnet_basis,
    df2_m1_tqnet_rfft,
    df2_h1_tqnet_basis,
    df2_h1_tqnet_rfft
], ignore_index=True)


df2_h1_pdf_basis = df2[
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.6) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.8) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 1.0) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.8) & (df2.learning_rate == 0.0005))
]

df2_h1_pdf_rfft = df2[
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 1.0) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.9) & (df2.learning_rate == 0.001)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 1.0) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.001))
]


df2_ecl_pdf_basis = df2[
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.999) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.999) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 0.8) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.999) & (df2.learning_rate == 0.0005))
]

df2_ecl_pdf_rfft = df2[
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 96)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.98) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 0.98) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.98) & (df2.learning_rate == 0.0002))
]


df2_ecl_pdf_kp = df2[
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'fourier_koopman') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'fourier_koopman') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'fourier_koopman') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'fourier_koopman') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.0005))
]

df2_ecl_pdf_dilate = df2[
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'dilate_cuda') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.05) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'dilate_cuda') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.15) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'dilate_cuda') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.0005) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode == 'dilate_cuda') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.1) & (df2.learning_rate == 0.0005))
]


df2_wea_pdf_basis = df2[
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.98) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.99) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 0.99) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'basis') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.99) & (df2.learning_rate == 0.0005))
]


df2_wea_pdf_rfft = df2[
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 192) &
    (df2.auxi_lambda == 0.1) & (df2.learning_rate == 0.0005)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & (df2.pred_len == 336) &
    (df2.auxi_lambda == 0.5) & (df2.learning_rate == 0.0002)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'rfft') & df2.pred_len.isin([720]) &
    (df2.auxi_lambda == 0.9) & (df2.learning_rate == 0.0002))
]

df2_wea_pdf_dilate = df2[
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'dilate_cuda') & (df2.pred_len == 96) &
    (df2.auxi_lambda == 0.4) & (df2.learning_rate == 0.005)) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode == 'dilate_cuda') & df2.pred_len.isin([192, 336, 720]))
]

df2_pdf_others = df2[
    ((df2.data_id.isin(['ETTm1'])) & (df2.model == 'PDF')) |
    (df2.data_id.isin(['ETTh1']) & (df2.model == 'PDF') & (df2.auxi_mode != 'basis') & (df2.auxi_mode != 'rfft')) |
    (df2.data_id.isin(['ECL']) & (df2.model == 'PDF') & (df2.auxi_mode != 'basis') & (df2.auxi_mode != 'rfft') & (df2.auxi_mode != 'fourier_koopman') & (df2.auxi_mode != 'dilate_cuda')) |
    (df2.data_id.isin(['Weather']) & (df2.model == 'PDF') & (df2.auxi_mode != 'basis') & (df2.auxi_mode != 'rfft') & (df2.auxi_mode != 'dilate_cuda'))
]

df2_pdf = pd.concat([
    df2_pdf_others,
    df2_h1_pdf_basis,
    df2_h1_pdf_rfft,
    df2_ecl_pdf_basis,
    df2_ecl_pdf_rfft,
    df2_ecl_pdf_kp,
    df2_ecl_pdf_dilate,
    df2_wea_pdf_basis,
    df2_wea_pdf_rfft,
    df2_wea_pdf_dilate
], ignore_index=True)

df2 = pd.concat([
    df2_tqnet,
    df2_pdf
], ignore_index=True)


columns = ['model', 'pred_len', 'data_id', 'auxi_mode', 'mse', 'mae']
df_loss = pd.concat([df2[columns], best[columns], base[columns]], ignore_index=True)
df_loss['pred_len'] = df_loss['pred_len'].astype(int)
df_loss.replace({'auxi_mode': {'basis': 'Time-o1', 'rfft': 'FreDF', 'fourier_koopman': 'Fourier Koopman', 'dpp': 'DPP', 'dilate_cuda': 'Dilate', 'soft_dtw': 'Soft-DTW', 'dtw': 'DTW', 'PDF': 'Time-o1', 'QDF': 'QDF'}}, inplace=True)

df_loss = df_loss[df_loss['auxi_mode'].isin(['DF', 'Time-o1', 'FreDF', 'Fourier Koopman', 'Dilate', 'Soft-DTW', 'QDF'])]

dst_order = ['ETTm1', 'ETTh1', 'ECL', 'Weather']
df_loss['data_id'] = pd.Categorical(df_loss['data_id'], categories=dst_order, ordered=True)

model_order = ['TQNet', 'PDF']
df_loss['model'] = pd.Categorical(df_loss['model'], categories=model_order, ordered=True)

mode_order = ['QDF', 'Time-o1', 'FreDF', 'Fourier Koopman', 'Dilate', 'Soft-DTW', 'DF']
df_loss['auxi_mode'] = pd.Categorical(df_loss['auxi_mode'], categories=mode_order, ordered=True)

dfl = df_loss.copy()

# save_root = '/data/home/Licheng/workspace/TSF-PCA/stats'
# save_cols = ['data_id', 'pred_len', 'mse', 'mae', 'auxi_mode']
# df_loss.round(3)[save_cols].to_csv(f'{save_root}/diff_loss.csv', index=False, float_format='%.3f')
dfl.head(5)

,model,pred_len,data_id,auxi_mode,mse,mae
0,TQNet,96,ECL,Time-o1,0.135795,0.228255
1,TQNet,96,ECL,FreDF,0.136414,0.228344
2,TQNet,96,ECL,Fourier Koopman,0.136640,0.231005
3,TQNet,96,ECL,Soft-DTW,0.162220,0.258020
4,TQNet,96,ECL,Dilate,0.136558,0.230732


In [36]:
df2_m1_rfft_ot


,model,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,lradj,train_epochs,patience,mask_factor,distance,joint_forecast,ot_type,batch_size,auxi_loss,reg_sk,eps,var_weight,auxi_type,auxi_mode,rank_ratio,reinit,use_weights,pca_dim,mse,mae,exp_dir
398,TimeBridge,720,ETTm1,0.005,0.0,1.0,type1,100,15,0.01,time,0,emd1d_h,128,MAE,0.1,1.000000e-09,1.0,complex,rfft,1.0,0,0,all,0.450374,0.437994,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


In [90]:
dfl[(dfl['model'] == 'TQNet') & (dfl['data_id'] == 'ETTh1') & (dfl['auxi_mode'] == 'FreDF')]
df2[(df2['model'] == 'TQNet') & (df2['data_id'] == 'ETTm1') & (df2['auxi_mode'] == 'rfft')]

,model,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,lradj,train_epochs,patience,mask_factor,distance,joint_forecast,ot_type,batch_size,auxi_loss,reg_sk,eps,var_weight,auxi_type,auxi_mode,rank_ratio,reinit,use_weights,pca_dim,mse,mae,exp_dir
8,TimeBridge,96,ETTm1,0.005,0.1,0.9,type1,100,15,0.01,time,0,emd1d_h,128,MAE,0.1,1.000000e-09,1.0,complex,rfft,1.0,0,0,all,0.321864,0.358082,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
9,TimeBridge,192,ETTm1,0.005,0.1,0.9,type1,100,15,0.01,time,0,emd1d_h,128,MAE,0.1,1.000000e-09,1.0,complex,rfft,1.0,0,0,all,0.366655,0.381490,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


In [187]:
df_loss = dfl.copy()

min_mse_idx = df_loss.groupby(['model', 'data_id', 'auxi_mode', 'pred_len'], observed=True)['mse'].idxmin()
df_loss = df_loss.loc[min_mse_idx]

df_loss_avg = df_loss.groupby(['model', 'data_id', 'auxi_mode']).mean(numeric_only=True).reset_index()
df_loss_avg['pred_len'] = 'Avg'
df_loss = pd.concat([df_loss, df_loss_avg]).reset_index(drop=True)
df_loss.sort_values(by=['model', 'data_id', 'auxi_mode', 'pred_len'], inplace=True)
df_loss = df_loss.reset_index(drop=True)
df_loss.dropna(inplace=True, thresh=6)

df_loss.head()

/tmp/ipykernel_1484514/1529525654.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_loss_avg = df_loss.groupby(['model', 'data_id', 'auxi_mode']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,auxi_mode,mse,mae
0,TQNet,96,ETTm1,QDF,0.306602,0.348971
1,TQNet,192,ETTm1,QDF,0.352414,0.376268
2,TQNet,336,ETTm1,QDF,0.382595,0.397512
3,TQNet,720,ETTm1,QDF,0.441163,0.434478
4,TQNet,Avg,ETTm1,QDF,0.370693,0.389307


In [188]:
df_loss_avg = df_loss[df_loss['pred_len'] == 'Avg'].copy()

df_loss_ = df_loss_avg.set_index(['model', 'data_id', 'auxi_mode']).unstack('auxi_mode').swaplevel(axis=1)
columns = []
for model in df_loss_.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df_loss_ = df_loss_[columns]

# save_root = '/data/home/Licheng/workspace/TSF-PCA/stats'
# df_loss_it.to_csv(f'{save_root}/diff_loss_itransformer.csv', index=True, float_format='%.3f')

df_loss_

auxi_mode           QDF             Time-o1               FreDF            \
                    mse       mae       mse       mae       mse       mae   
model data_id                                                               
TQNet ETTm1    0.370693  0.389307  0.372338  0.389944  0.374751  0.390170   
      ETTh1    0.431247  0.431361  0.437044  0.432206  0.432138  0.431727   
      ECL      0.164594  0.256630  0.167179  0.257164  0.167993  0.257312   
      Weather  0.242499  0.267768  0.244989  0.269440  0.244090  0.268341   
PDF   ETTm1    0.380658  0.393839  0.386285  0.398759  0.387191  0.400413   
      ETTh1    0.435861  0.429496  0.437621  0.437983  0.436893  0.435182   
      ECL      0.193681  0.276956  0.194826  0.276002  0.193689  0.274405   
      Weather  0.259451  0.281072  0.263599  0.284311  0.267907  0.287135   

auxi_mode     Fourier Koopman              Dilate            Soft-DTW  \
                          mse       mae       mse       mae       mse   
model data_id                                                           
TQNet ETTm1          0.595413  0.499368  0.376015  0.391397  0.387269   
      ETTh1          0.450521  0.441584  0.454954  0.442024  0.453474   
      ECL            0.166425  0.258396  0.167032  0.258815  0.622900   
      Weather        0.282417  0.305871  0.245663  0.270328  0.255190   
PDF   ETTm1          0.587038  0.485043  0.388913  0.403669  0.395555   
      ETTh1          0.496853  0.472400  0.436856  0.440351  0.446803   
      ECL            0.195997  0.280922  0.188339  0.273377  0.695053   
      Weather        0.267524  0.289500  0.261767  0.284187  1.296263   

auxi_mode                      DF            
                    mae       mse       mae  
model data_id                                
TQNet ETTm1    0.394336  0.376018  0.391403  
      ETTh1    0.438345  0.448900  0.438991  
      ECL      0.524233  0.174991  0.265458  
      Weather  0.275603  0.245663  0.270328  
PDF   ETTm1    0.404039  0.386515  0.395981  
      ETTh1    0.446676  0.451670  0.439646  
      ECL      0.547779  0.197936  0.280646  
      Weather  0.452284  0.264530  0.282979

## write to table

In [101]:


contents = []
for model in ['TQNet', 'PDF']:
    line = r"\multirow{4}{*}{{\rotatebox{90}{\scaleb{" + model + r"}}}}"
    contents.append(line)
    for data_id in ['ETTm1', 'ETTh1', 'ECL', 'Weather']:
        _df = df_loss_avg[(df_loss_avg['model'] == model) & (df_loss_avg['data_id'] == data_id)]
        line = f"& {data_id} "
        for i, row in enumerate(_df.itertuples()):
            line += f"& {row.mse:.3f} & {row.mae:.3f} "
        line += r"\\"
        contents.append(line)
    if model != 'PDF':
        contents.append(r"\midrule")

print("\n".join(contents))

\multirow{4}{*}{{\rotatebox{90}{\scaleb{TQNet}}}}
& ETTm1 & 0.371 & 0.389 & 0.371 & 0.388 & 0.371 & 0.388 & 0.595 & 0.499 & 0.376 & 0.391 & 0.387 & 0.394 & 0.376 & 0.391 \\
& ETTh1 & 0.431 & 0.431 & 0.437 & 0.432 & 0.432 & 0.432 & 0.451 & 0.442 & 0.455 & 0.442 & 0.453 & 0.438 & 0.449 & 0.439 \\
& ECL & 0.165 & 0.257 & 0.167 & 0.257 & 0.168 & 0.257 & 0.166 & 0.258 & 0.167 & 0.259 & 0.623 & 0.524 & 0.175 & 0.265 \\
& Weather & 0.242 & 0.268 & 0.245 & 0.269 & 0.244 & 0.268 & 0.282 & 0.306 & 0.246 & 0.270 & 0.255 & 0.276 & 0.246 & 0.270 \\
\midrule
\multirow{4}{*}{{\rotatebox{90}{\scaleb{PDF}}}}
& ETTm1 & 0.381 & 0.394 & 0.386 & 0.399 & 0.387 & 0.400 & 0.587 & 0.485 & 0.389 & 0.404 & 0.396 & 0.404 & 0.387 & 0.396 \\
& ETTh1 & 0.436 & 0.429 & 0.438 & 0.438 & 0.437 & 0.435 & 0.497 & 0.472 & 0.437 & 0.440 & 0.447 & 0.447 & 0.452 & 0.440 \\
& ECL & 0.194 & 0.277 & 0.192 & 0.273 & 0.196 & 0.276 & 0.196 & 0.281 & 0.188 & 0.274 & 0.695 & 0.548 & 0.198 & 0.281 \\
& Weather & 0.259 & 0.281 & 0.264 

In [189]:
import numpy as np

contents = []
for model in ['TQNet', 'PDF']:
    line = r"\multirow{4}{*}{{\rotatebox{90}{\scaleb{" + model + r"}}}}"
    contents.append(line)
    for data_id in ['ETTm1', 'ETTh1', 'ECL', 'Weather']:
        _df = df_loss_avg[(df_loss_avg['model'] == model) & (df_loss_avg['data_id'] == data_id)]
        _df = _df[_df.auxi_mode != 'Dilate']
        
        # 找出MSE和MAE的最小值和次最小值
        mse_values = _df['mse'].values
        mae_values = _df['mae'].values
        
        mse_min = np.min(mse_values)
        mae_min = np.min(mae_values)
        
        # 找次最小值：排除最小值后的最小值
        mse_second_min = np.min(mse_values[mse_values > mse_min]) if np.any(mse_values > mse_min) else None
        mae_second_min = np.min(mae_values[mae_values > mae_min]) if np.any(mae_values > mae_min) else None
        
        line = f"& {data_id} "
        for i, row in enumerate(_df.itertuples()):
            # 处理MSE
            if row.mse == mse_min:
                mse_str = f"\\bst{{{row.mse:.3f}}}"
            elif mse_second_min is not None and row.mse == mse_second_min:
                mse_str = f"\\subbst{{{row.mse:.3f}}}"
            else:
                mse_str = f"{row.mse:.3f}"
            
            # 处理MAE
            if row.mae == mae_min:
                mae_str = f"\\bst{{{row.mae:.3f}}}"
            elif mae_second_min is not None and row.mae == mae_second_min:
                mae_str = f"\\subbst{{{row.mae:.3f}}}"
            else:
                mae_str = f"{row.mae:.3f}"
            
            line += f"& {mse_str} & {mae_str} "
        line += r"\\"
        contents.append(line)
    if model != 'PDF':
        contents.append(r"\midrule")

print("\n".join(contents))


\multirow{4}{*}{{\rotatebox{90}{\scaleb{TQNet}}}}
& ETTm1 & \bst{0.371} & \bst{0.389} & \subbst{0.372} & \subbst{0.390} & 0.375 & 0.390 & 0.595 & 0.499 & 0.387 & 0.394 & 0.376 & 0.391 \\
& ETTh1 & \bst{0.431} & \bst{0.431} & 0.437 & 0.432 & \subbst{0.432} & \subbst{0.432} & 0.451 & 0.442 & 0.453 & 0.438 & 0.449 & 0.439 \\
& ECL & \bst{0.165} & \bst{0.257} & 0.167 & \subbst{0.257} & 0.168 & 0.257 & \subbst{0.166} & 0.258 & 0.623 & 0.524 & 0.175 & 0.265 \\
& Weather & \bst{0.242} & \bst{0.268} & 0.245 & 0.269 & \subbst{0.244} & \subbst{0.268} & 0.282 & 0.306 & 0.255 & 0.276 & 0.246 & 0.270 \\
\midrule
\multirow{4}{*}{{\rotatebox{90}{\scaleb{PDF}}}}
& ETTm1 & \bst{0.381} & \bst{0.394} & \subbst{0.386} & 0.399 & 0.387 & 0.400 & 0.587 & 0.485 & 0.396 & 0.404 & 0.387 & \subbst{0.396} \\
& ETTh1 & \bst{0.436} & \bst{0.429} & 0.438 & 0.438 & \subbst{0.437} & \subbst{0.435} & 0.497 & 0.472 & 0.447 & 0.447 & 0.452 & 0.440 \\
& ECL & \bst{0.194} & 0.277 & 0.195 & \subbst{0.276} & \subbst{0.194} &

## write to full table

In [196]:
contents = []

model = 'TQNet'
model = 'PDF'

df = df_loss[(df_loss['model'] == model) & (df_loss.auxi_mode != 'Dilate')].copy()

for data_id in ['ETTm1', 'ETTh1', 'ECL', 'Weather']:
    line = r"\multirow{5}{*}{{\rotatebox{90}{\scalebox{0.95}{" + data_id + r"}}}}"
    contents.append(line)
    for pl in [96, 192, 336, 720, 'Avg']:
        _df = df[(df['data_id'] == data_id) & (df['pred_len'] == pl)]
        line = f"& {pl} "
        for row in _df.itertuples():
            line += f"& {row.mse:.3f} & {row.mae:.3f} "
        line += r"\\"
        contents.append(line)
        if pl == 720:
            contents.append(r"\cmidrule(lr){2-14}")
    if data_id != 'Weather':
        contents.append(r"\midrule")

print("\n".join(contents))

\multirow{5}{*}{{\rotatebox{90}{\scalebox{0.95}{ETTm1}}}}
& 96 & 0.320 & 0.358 & 0.326 & 0.361 & 0.325 & 0.362 & 1.051 & 0.663 & 0.323 & 0.362 & 0.326 & 0.363 \\
& 192 & 0.361 & 0.380 & 0.371 & 0.386 & 0.372 & 0.388 & 0.420 & 0.414 & 0.371 & 0.388 & 0.365 & 0.381 \\
& 336 & 0.390 & 0.401 & 0.401 & 0.409 & 0.399 & 0.409 & 0.421 & 0.415 & 0.408 & 0.413 & 0.397 & 0.402 \\
& 720 & 0.451 & 0.437 & 0.448 & 0.439 & 0.453 & 0.443 & 0.456 & 0.448 & 0.480 & 0.454 & 0.458 & 0.437 \\
\cmidrule(lr){2-14}
& Avg & 0.381 & 0.394 & 0.386 & 0.399 & 0.387 & 0.400 & 0.587 & 0.485 & 0.396 & 0.404 & 0.387 & 0.396 \\
\midrule
\multirow{5}{*}{{\rotatebox{90}{\scalebox{0.95}{ETTh1}}}}
& 96 & 0.375 & 0.391 & 0.380 & 0.403 & 0.373 & 0.393 & 0.632 & 0.533 & 0.383 & 0.405 & 0.388 & 0.400 \\
& 192 & 0.423 & 0.419 & 0.422 & 0.425 & 0.423 & 0.426 & 0.424 & 0.429 & 0.430 & 0.432 & 0.440 & 0.428 \\
& 336 & 0.461 & 0.439 & 0.463 & 0.441 & 0.477 & 0.446 & 0.456 & 0.450 & 0.462 & 0.453 & 0.483 & 0.449 \\
& 720 & 0.484 & 0